# IT 데이터 추가 수집 (파이프라인 통일용)

**이 노트북이 하는 일**: 통일 파이프라인에 넣을 IT 데이터를 **공통 형식 CSV 3개**로 만들어 저장합니다. 분석은 하지 않습니다. 분석은 통일 파이프라인(공통 코드)에서 합니다.

**왜 다시 받나요?** 통일 규칙대로 계산하려면 2022년보다 더 옛날 데이터가 필요합니다.

| 쓰는 곳 | 필요한 데이터 |
|---|---|
| 회사 기준 점수 (직전 8분기) | 2022년 초 발표의 "평소"를 계산하려면 2020~2021년 실적과 주가 |
| 대조군 판정 | 2022년 1월 발표의 대조군을 고르려면 2021년 말 발표 기록 |
| 평소 거래량, 플라시보 | 2021년 중반부터의 주가 |

**기존 IT 노트북에서 바뀐 점**

| 항목 | 기존 | 이번 |
|---|---|---|
| 주가 시작 | 2021-07 | **2020-01** |
| 주가 끝 | 2026-10-06 | **2026-10-20** |
| 종가 | 분할·배당 조정 가격만 | **분할만 조정한 종가(`close`) + 분할·배당 조정 종가(`adjclose`) 둘 다** |
| 시장 기준 | S&P500 지수(^GSPC) | **SPY** |
| 업종 ETF | 없음 | **XLK** (강건성용) |
| 실적 | 2020년부터 있음 | 그대로, 공통 형식으로 다시 저장 |

`close`는 SUE를 계산할 때 주가로 나누는 데 쓰고, `adjclose`는 수익률을 계산하는 데 씁니다.

**회사 목록**은 기존 IT 파일럿과 같은 30개사입니다. 2022년 초 추정 시가총액 상위 30개(반도체 제외)를 그대로 고정했습니다.

**저장되는 파일** (`data/it/` 폴더)

| 파일 | 한 줄 = | 컬럼 |
|---|---|---|
| `companies_it.csv` | 회사 1개 | `ticker, name, subgroup` |
| `earnings_it.csv` | 실적발표 1번 | `ticker, announce_datetime, timing, eps_actual, eps_consensus` |
| `prices_it.csv` | 종목 1개 × 거래일 1일 | `date, ticker, close, adjclose, volume` |

## 1. 준비

처음 한 번만 아래 `# !pip install` 줄의 `#`을 지우고 실행하세요.

코랩에서 결과를 드라이브에 남기려면 `USE_DRIVE = True`로 바꾸세요. 그러면 `DRIVE_FOLDER` 안에 `data/it/`가 만들어집니다.

In [1]:
# !pip install yfinance

USE_DRIVE = False
DRIVE_FOLDER = "/content/drive/MyDrive/did_pilot"

import os, time
from pathlib import Path
import numpy as np
import pandas as pd
import yfinance as yf

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(DRIVE_FOLDER, exist_ok=True)
    os.chdir(DRIVE_FOLDER)

OUT = Path("data/it")
OUT.mkdir(parents=True, exist_ok=True)

START = "2020-01-01"     # 주가·실적 시작
END = "2026-10-21"       # 이 날짜는 포함되지 않아서 실제로는 2026-10-20까지
ETFS = ["SPY", "XLK"]    # 시장 기준(SPY), IT 업종 ETF(XLK)
print("yfinance", yf.__version__, "/ 저장 위치:", OUT.resolve())

yfinance 0.2.66 / 저장 위치: /content/data/it


## 2. 회사 목록 (기존 IT 파일럿 30개사 고정)

`subgroup`은 IT 안의 세부업종 묶음입니다. 대조군을 "같은 하위그룹 / 다른 하위그룹"으로 나눠보는 강건성 분석에 씁니다.

In [2]:
COMPANIES = [
    ("AAPL", "Apple", "Hardware & Storage"), ("MSFT", "Microsoft", "Software"),
    ("ORCL", "Oracle", "Software"), ("ACN", "Accenture", "IT Services"),
    ("ADBE", "Adobe", "Software"), ("CSCO", "Cisco", "Comm Equipment & Optical"),
    ("CRM", "Salesforce", "Software"), ("INTU", "Intuit", "Software"),
    ("NOW", "ServiceNow", "Software"), ("IBM", "IBM", "IT Services"),
    ("PANW", "Palo Alto Networks", "Software"), ("SNPS", "Synopsys", "Software"),
    ("WDAY", "Workday", "Software"), ("ADSK", "Autodesk", "Software"),
    ("DDOG", "Datadog", "Software"), ("CRWD", "CrowdStrike", "Software"),
    ("APH", "Amphenol", "Electronics"), ("CDNS", "Cadence Design Systems", "Software"),
    ("FTNT", "Fortinet", "Software"), ("ANET", "Arista Networks", "Comm Equipment & Optical"),
    ("ROP", "Roper Technologies", "Electronics"), ("PLTR", "Palantir Technologies", "Software"),
    ("TEL", "TE Connectivity", "Electronics"), ("MSI", "Motorola Solutions", "Comm Equipment & Optical"),
    ("CTSH", "Cognizant", "IT Services"), ("KEYS", "Keysight Technologies", "Electronics"),
    ("DELL", "Dell Technologies", "Hardware & Storage"), ("HPQ", "HP Inc.", "Hardware & Storage"),
    ("GLW", "Corning", "Electronics"), ("ZBRA", "Zebra Technologies", "Electronics"),
]
companies = pd.DataFrame(COMPANIES, columns=["ticker", "name", "subgroup"])
tickers = companies["ticker"].tolist()
companies.to_csv(OUT / "companies_it.csv", index=False)
companies["subgroup"].value_counts()

,count
subgroup,
Software,15
Electronics,6
Hardware & Storage,3
IT Services,3
Comm Equipment & Optical,3


## 3. 주가·거래량

`auto_adjust=False`로 받아야 두 종류의 종가가 다 나옵니다.
- `close`: 주식 분할만 반영한 종가. SUE 분모용
- `adjclose`: 분할과 배당을 모두 반영한 종가. 수익률 계산용

예: 애플이 배당을 주면 `adjclose`는 과거 가격을 그만큼 살짝 낮춰서 "배당까지 받았을 때의 수익률"이 맞게 나오도록 합니다. `close`는 그대로 둡니다.

In [3]:
raw = yf.download(tickers + ETFS, start=START, end=END, auto_adjust=False, progress=False)

parts = []
for col, name in [("Close", "close"), ("Adj Close", "adjclose"), ("Volume", "volume")]:
    s = raw[col].reset_index().melt(id_vars="Date", var_name="ticker", value_name=name)
    parts.append(s.set_index(["Date", "ticker"]))
prices = (pd.concat(parts, axis=1).reset_index()
          .rename(columns={"Date": "date"})
          .dropna(subset=["close"])
          .sort_values(["ticker", "date"]).reset_index(drop=True))
prices["date"] = pd.to_datetime(prices["date"]).dt.strftime("%Y-%m-%d")
prices.to_csv(OUT / "prices_it.csv", index=False)

print(f"주가 {len(prices):,}줄 / 종목 {prices['ticker'].nunique()}개 (회사 {len(tickers)} + ETF {len(ETFS)})")
print("전체 기간:", prices["date"].min(), "~", prices["date"].max())

주가 54,500줄 / 종목 32개 (회사 30 + ETF 2)
전체 기간: 2020-01-02 ~ 2026-10-20


**주가 점검**: 종목별 첫날과 마지막 날입니다.
- 대부분 `2020-01-02`에 시작해야 정상입니다.
- 늦게 시작하는 회사는 그때 상장했거나 분사한 회사입니다. 예: 팔란티어(PLTR)는 2020년 9월 상장이라 그때부터 시작합니다.
- 마지막 날이 `2026-10-20` 근처가 아니면 데이터가 덜 받아진 것이니 다시 실행하세요.

In [4]:
span = prices.groupby("ticker")["date"].agg(first="min", last="max", n_days="count")
late = span[span["first"] > "2020-01-10"]
print("늦게 시작하는 종목:", late.to_dict("index") if len(late) else "없음")
print("마지막 날이 2026-10-15보다 이른 종목:", span[span["last"] < "2026-10-15"].index.tolist() or "없음")
span

늦게 시작하는 종목: {'PLTR': {'first': '2020-09-30', 'last': '2026-10-20', 'n_days': 1521}}
마지막 날이 2026-10-15보다 이른 종목: 없음


,first,last,n_days
ticker,,,
AAPL,2020-01-02,2026-10-20,1709
ACN,2020-01-02,2026-10-20,1709
ADBE,2020-01-02,2026-10-20,1709
ADSK,2020-01-02,2026-10-20,1709
ANET,2020-01-02,2026-10-20,1709
APH,2020-01-02,2026-10-20,1709
CDNS,2020-01-02,2026-10-20,1709
CRM,2020-01-02,2026-10-20,1709
CRWD,2020-01-02,2026-10-20,1709


## 4. 실적 (2020년부터)

`get_earnings_dates(limit=40)`는 회사마다 최근 약 10년치 발표 기록을 줍니다. 그중 2020년 이후만 남깁니다.

**발표 시각 분류** (뉴욕 시간)
- 9:30 이전 → `BMO` (장 시작 전)
- 16:00 이후 → `AMC` (장 마감 후)
- 그 사이 → `during` (장중)
- 00:00으로 기록 → `unknown` (시각 모름. 통일 규칙상 분석에서 제외 예정)

반응일(t0)은 통일 파이프라인에서 이 `timing`을 보고 계산합니다.

In [5]:
def get_earnings(ticker, limit=40):
    d = yf.Ticker(ticker).get_earnings_dates(limit=limit)
    if d is None or len(d) == 0:
        return pd.DataFrame()
    d = d.reset_index()
    ren = {d.columns[0]: "announce_datetime"}
    for c in d.columns[1:]:
        lc = str(c).lower()
        if "reported" in lc:
            ren[c] = "eps_actual"
        elif "estimate" in lc:
            ren[c] = "eps_consensus"
    d = d.rename(columns=ren)
    d["ticker"] = ticker
    return d[["ticker", "announce_datetime", "eps_actual", "eps_consensus"]]

rows, failed = [], []
for t in tickers:
    try:
        d = get_earnings(t)
        rows.append(d) if len(d) else failed.append(t)
    except Exception as e:
        failed.append(t); print(t, "실패:", repr(e)[:100])
    time.sleep(1.0)

earn = pd.concat(rows, ignore_index=True)
ts = pd.to_datetime(earn["announce_datetime"], utc=True).dt.tz_convert("America/New_York")
h = ts.dt.hour + ts.dt.minute / 60
earn["timing"] = np.select([h == 0, h < 9.5, h >= 16], ["unknown", "BMO", "AMC"], "during")
earn["announce_datetime"] = ts.dt.strftime("%Y-%m-%d %H:%M:%S%z")
earn["eps_actual"] = pd.to_numeric(earn["eps_actual"], errors="coerce")
earn["eps_consensus"] = pd.to_numeric(earn["eps_consensus"], errors="coerce")
keep = earn["eps_actual"].notna() & (ts >= pd.Timestamp(START, tz="America/New_York"))
earn = (earn[keep][["ticker", "announce_datetime", "timing", "eps_actual", "eps_consensus"]]
        .drop_duplicates(subset=["ticker", "announce_datetime"])
        .sort_values(["ticker", "announce_datetime"]).reset_index(drop=True))
earn.to_csv(OUT / "earnings_it.csv", index=False)

print(f"실적 {len(earn)}건 / 실패한 회사: {failed or '없음'}")
print("발표 시각:", earn["timing"].value_counts().to_dict())

실적 822건 / 실패한 회사: 없음
발표 시각: {'AMC': 614, 'BMO': 205, 'during': 3}


**실적 점검**: 2020년 1월부터 지금까지면 회사당 **약 27분기**가 정상입니다.
- `first`가 2020년 초가 아니면, 그 회사는 2022년 초 발표의 "직전 8분기"가 모자랄 수 있습니다. 팔란티어처럼 2020년 하반기에 상장한 회사는 늦게 시작하는 게 정상입니다.
- `consensus_missing`이 0보다 크면 예상 EPS가 빈 분기가 있다는 뜻입니다.

In [6]:
ets = pd.to_datetime(earn["announce_datetime"], utc=True)
cov = (earn.assign(d=ets.dt.strftime("%Y-%m-%d"))
       .groupby("ticker").agg(n=("d", "size"), first=("d", "min"), last=("d", "max"),
                              consensus_missing=("eps_consensus", lambda s: int(s.isna().sum()))))
print(f"회사당 평균 {cov['n'].mean():.1f}분기, 최소 {cov['n'].min()}분기")
cov.sort_values("n")

회사당 평균 27.4분기, 최소 25분기


,n,first,last,consensus_missing
ticker,,,,
PLTR,25,2020-11-12,2026-11-02,0
CRWD,26,2020-06-02,2026-08-26,2
ADSK,27,2020-02-27,2026-08-27,0
ACN,27,2020-03-19,2026-10-01,0
DELL,27,2020-02-27,2026-09-01,0
HPQ,27,2020-02-24,2026-08-26,0
CSCO,27,2020-02-12,2026-08-12,0
ADBE,27,2020-03-12,2026-09-10,0
KEYS,27,2020-02-24,2026-08-18,0


## 5. 확인: 주식 분할한 회사의 EPS가 분할 조정돼 있나요?

SUE는 (실제 EPS − 예상 EPS) ÷ 주가(`close`)입니다. `close`는 분할이 반영돼 있으니까 **EPS도 분할이 반영돼 있어야** 단위가 맞습니다.

**읽는 법**: 아래에 2020년 이후 주식을 분할한 회사의 EPS가 날짜순으로 나옵니다. 분할 날짜 앞뒤로
- EPS가 비슷한 크기로 이어지면 → **분할 조정돼 있음 (정상)**
- EPS가 분할 비율만큼 갑자기 뛰거나 떨어지면 → **조정 안 됨**. 예: 2대1 분할인데 분할 전 EPS가 분할 후보다 2배쯤 크다면 문제

마지막 줄의 판정은 "분할 직전 4분기 평균 EPS ÷ 분할 직후 4분기 평균 EPS"를 분할 비율과 비교한 대략적인 기준입니다. 판정이 애매하면 표를 직접 보고 판단하세요.

In [7]:
split_rows = []
for t in tickers:
    sp = yf.Ticker(t).splits
    if sp is None or len(sp) == 0:
        continue
    sp.index = pd.to_datetime(sp.index, utc=True)
    sp = sp[sp.index >= pd.Timestamp(START, tz="UTC")]
    for when, ratio in sp.items():
        e = earn[earn.ticker == t].assign(dt=pd.to_datetime(earn.loc[earn.ticker == t, "announce_datetime"], utc=True))
        before = e[e.dt < when].tail(4)["eps_actual"].mean()
        after = e[e.dt >= when].head(4)["eps_actual"].mean()
        r = before / after if after else np.nan
        verdict = ("조정 안 된 듯 (확인 필요)" if pd.notna(r) and abs(r - ratio) < abs(r - 1) else "조정된 듯 (정상)")
        split_rows.append(dict(ticker=t, split_date=when.date(), ratio=ratio,
                               eps_before_4q=round(before, 3), eps_after_4q=round(after, 3), verdict=verdict))
        print(f"\n{t}: {when.date()} {ratio:g}대1 분할")
        print(e[(e.dt >= when - pd.Timedelta(days=400)) & (e.dt <= when + pd.Timedelta(days=400))]
              [["announce_datetime", "eps_consensus", "eps_actual"]].to_string(index=False))

split_check = pd.DataFrame(split_rows)
split_check.to_csv(OUT / "split_check_it.csv", index=False)
split_check if len(split_check) else print("2020년 이후 분할한 회사 없음")


AAPL: 2020-08-31 4대1 분할
       announce_datetime  eps_consensus  eps_actual
2020-01-28 16:00:00-0500           1.13        1.25
2020-04-30 17:00:00-0400           0.56        0.64
2020-07-30 16:00:00-0400           0.52        0.65
2020-10-29 16:00:00-0400           0.70        0.73
2021-01-27 16:00:00-0500           1.42        1.68
2021-04-28 16:00:00-0400           0.98        1.40
2021-07-27 16:00:00-0400           1.01        1.30

NOW: 2025-12-18 5대1 분할
       announce_datetime  eps_consensus  eps_actual
2025-01-29 16:00:00-0500           0.73        0.73
2025-04-23 16:00:00-0400           0.77        0.81
2025-07-23 16:00:00-0400           0.71        0.82
2025-10-29 16:00:00-0400           0.85        0.96
2026-01-28 16:00:00-0500           0.89        0.92
2026-04-22 16:00:00-0400           0.97        0.97
2026-07-22 16:00:00-0400           0.86        0.90
2026-10-28 16:00:00-0400           1.02        1.00

IBM: 2021-11-04 1.046대1 분할
       announce_datetime  eps_consensus

,ticker,split_date,ratio,eps_before_4q,eps_after_4q,verdict
0,AAPL,2020-08-31,4.000,0.847,1.278,조정된 듯 (정상)
1,NOW,2025-12-18,5.000,0.830,0.948,조정된 듯 (정상)
2,IBM,2021-11-04,1.046,2.172,2.218,조정된 듯 (정상)
3,PANW,2022-09-14,3.000,0.315,0.548,조정된 듯 (정상)
4,PANW,2024-12-16,2.000,0.733,0.872,조정된 듯 (정상)
5,CRWD,2026-07-02,4.000,0.258,0.310,조정된 듯 (정상)
6,APH,2021-03-05,2.000,0.118,0.155,조정된 듯 (정상)
7,APH,2024-06-12,2.000,0.198,0.268,조정된 듯 (정상)
8,APH,2026-09-03,2.000,0.542,0.800,조정된 듯 (정상)
9,FTNT,2022-06-23,5.000,0.208,0.338,조정된 듯 (정상)


## 6. 마무리 체크리스트

1. `data/it/` 안에 아래 파일이 있는지 확인
   - `companies_it.csv`, `earnings_it.csv`, `prices_it.csv` (공통 파이프라인 입력)
   - `split_check_it.csv` (분할 확인 결과, 공유용)
2. 세 CSV를 레포의 `data/it/`에 올리기
3. 공통 파이프라인 담당에게 컬럼 이름 공유
   - 실적: `ticker, announce_datetime, timing, eps_actual, eps_consensus`
   - 주가: `date, ticker, close, adjclose, volume` (SPY, XLK 포함)
4. 분할 확인 결과 한 줄 공유 (전부 정상인지, 확인이 필요한 회사가 있는지)

In [8]:
for f in ["companies_it.csv", "earnings_it.csv", "prices_it.csv", "split_check_it.csv"]:
    p = OUT / f
    print(f"{'OK ' if p.exists() else '없음'} {p}  ({p.stat().st_size/1024:,.0f} KB)" if p.exists() else f"없음 {p}")

OK  data/it/companies_it.csv  (1 KB)
OK  data/it/earnings_it.csv  (35 KB)
OK  data/it/prices_it.csv  (3,280 KB)
OK  data/it/split_check_it.csv  (1 KB)
